# 🧠 Deep Learning untuk Analisis Sentimen Review Film (IMDB)

**Workshop Pemrosesan Bahasa Alami (PBA)**  
*Institut Teknologi Sumatera (ITERA)*  

---

## 🎯 Tujuan

Pada proyek ini, kita akan membangun dan membandingkan **dua model deep learning** untuk mengklasifikasikan sentimen review film menjadi:

- **Positive**
- **Negative**

---

## 🤖 Model yang Digunakan

| # | Model | Konsep Utama |
|--|------|-------------|
| 1 | **BiLSTM** | Recurrent neural network dua arah |
| 2 | **BiLSTM + Attention** | Fokus pada kata penting dalam teks |

---

## 📊 Dataset

- Dataset: **IMDB Movie Reviews (50K)**
- Sumber: https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews  
- Digunakan: **Subset (±3.000 – 10.000 data)**

---

## 🧾 Label

- **positive → 1**
- **negative → 0**

---

## ⚙️ Framework

- PyTorch
- Scikit-learn
- Pandas

---

## 🏗️ Arsitektur Model

### 🔹 BiLSTM

## 📘 1. Import Library
Penjelasan

Mengimpor semua library dan module yang dibutuhkan untuk preprocessing, training, dan evaluasi model.

In [9]:
import pandas as pd
import torch

from config import *
from preprocess import preprocess
from dataset import Vocabulary, get_lstm_dataloaders
from models import BiLSTMClassifier, BiLSTMAttentionClassifier
from train import set_seed, train_model, evaluate

## 📘 2. Set Seed & Device
📌 Penjelasan

Digunakan untuk memastikan hasil eksperimen konsisten dan menentukan apakah menggunakan CPU atau GPU.

In [10]:
set_seed(RANDOM_SEED)
print("Device:", DEVICE)

Device: cpu


## 📘 3. Load & Preprocess Dataset
📌 Penjelasan

Dataset IMDB (50K review) dimuat lalu diambil subset (±10K/3K sesuai config). Teks dibersihkan (cleaning) dan label di-encode menjadi angka.

In [11]:
df = preprocess()

print("Jumlah data:", len(df))
df.head()

📥 Load dataset...
PATH RAW: c:\Users\USER\Documents\Tubes NLP\pba2026-kelompok-3\DL\Data\imdb-dataset-of-50k-movie-reviews\IMDB Dataset.csv
EXIST: True
📊 Jumlah data awal: 50,000
🔀 Setelah sampling: 10,000
🧹 Cleaning text...
🔢 Encoding label...
📊 Setelah cleaning: 10,000
✅ Data bersih disimpan di: c:\Users\USER\Documents\Tubes NLP\pba2026-kelompok-3\DL\Data\clean_imdb_10k.csv
Jumlah data: 10000


,review,sentiment,clean_review,label_encoded
0,This was obviously the worst movie ever made.....,negative,this was obviously the worst movie ever made k...,0
1,Fame did something odd. It was not only a musi...,negative,fame did something odd it was not only a music...,0
2,"From all the rave reviews, we couldn't wait to...",negative,from all the rave reviews we couldn t wait to ...,0
3,"First, let me confess that I have not read thi...",positive,first let me confess that i have not read this...,1
4,Mickey Rourke is enjoying a renaissance at the...,negative,mickey rourke is enjoying a renaissance at the...,0


## 📘 4. Build Vocabulary
📌 Penjelasan

Mengubah kata menjadi angka (tokenisasi sederhana) agar bisa diproses oleh model deep learning.

In [12]:
vocab = Vocabulary()
vocab.build_vocab(df["clean_review"].tolist(), max_size=VOCAB_SIZE)

print("Ukuran vocab:", len(vocab.word2idx))

import json
from config import VOCAB_PATH, LABEL_ENCODER_PATH, LABEL_MAPPING

# save vocab
with open(VOCAB_PATH, "w") as f:
    json.dump(vocab.word2idx, f)

# save label mapping
with open(LABEL_ENCODER_PATH, "w") as f:
    json.dump(LABEL_MAPPING, f)

print("✅ vocab & label encoder saved!")

📖 Vocabulary: 5,000 kata
Ukuran vocab: 5000
✅ vocab & label encoder saved!


## 📘 5. Split Data & DataLoader
📌 Penjelasan

Dataset dibagi menjadi:

Train
Validation
Test

Kemudian dibuat DataLoader untuk proses training.

In [13]:
train_loader, val_loader, test_loader = get_lstm_dataloaders(df, vocab)

📦 Train: 6999 | Val: 1000 | Test: 2001


## 📘 6. Training Model 1 — BiLSTM
📌 Penjelasan

Model BiLSTM digunakan untuk menangkap konteks kata dari dua arah (forward & backward).

In [14]:
model_lstm = BiLSTMClassifier(
    vocab_size=len(vocab)
)
history_lstm = train_model(
    model_lstm,
    train_loader,
    val_loader,
    test_loader,
    save_path=BILSTM_MODEL_PATH,
    epochs=LSTM_EPOCHS,
    lr=LSTM_LR,
    patience=LSTM_PATIENCE,
    model_name="bilstm"   # 
)


🚀 Training bilstm...

Epoch 1
Train Loss: 0.6934 | Acc: 0.5188
Val   Loss: 0.6871 | Acc: 0.5480
Epoch 2
Train Loss: 0.6796 | Acc: 0.5679
Val   Loss: 0.6754 | Acc: 0.5630
Epoch 3
Train Loss: 0.6581 | Acc: 0.6095
Val   Loss: 0.6309 | Acc: 0.6480

📊 Evaluasi bilstm...

📄 Classification Report:
              precision    recall  f1-score   support

    negative       0.60      0.75      0.67      1001
    positive       0.67      0.50      0.57      1000

    accuracy                           0.63      2001
   macro avg       0.63      0.63      0.62      2001
weighted avg       0.63      0.63      0.62      2001

Accuracy: 0.6256871564217891
F1 Score: 0.6199663700913076
📊 Saved: c:\Users\USER\Documents\Tubes NLP\pba2026-kelompok-3\DL\plots\confusion_matrix_bilstm.png
📊 Saved: c:\Users\USER\Documents\Tubes NLP\pba2026-kelompok-3\DL\plots\training_curves_bilstm.png


## 📘 7. Training Model 2 — BiLSTM + Attention
📌 Penjelasan

Menambahkan mekanisme attention agar model fokus pada kata penting dalam kalimat.

In [15]:
model_att = BiLSTMAttentionClassifier(
    vocab_size=len(vocab)
)
history_att = train_model(
    model_att,
    train_loader,
    val_loader,
    test_loader,
    save_path=BILSTM_ATT_MODEL_PATH,
    epochs=LSTM_EPOCHS,
    lr=LSTM_LR,
    patience=LSTM_PATIENCE,
    model_name="bilstm_attention"
)


🚀 Training bilstm_attention...

Epoch 1
Train Loss: 0.6855 | Acc: 0.5589
Val   Loss: 0.6689 | Acc: 0.6150
Epoch 2
Train Loss: 0.6379 | Acc: 0.6378
Val   Loss: 0.6431 | Acc: 0.6590
Epoch 3
Train Loss: 0.5840 | Acc: 0.6911
Val   Loss: 0.5465 | Acc: 0.7200

📊 Evaluasi bilstm_attention...

📄 Classification Report:
              precision    recall  f1-score   support

    negative       0.71      0.69      0.70      1001
    positive       0.70      0.72      0.71      1000

    accuracy                           0.71      2001
   macro avg       0.71      0.71      0.71      2001
weighted avg       0.71      0.71      0.71      2001

Accuracy: 0.7056471764117941
F1 Score: 0.7055695250583867
📊 Saved: c:\Users\USER\Documents\Tubes NLP\pba2026-kelompok-3\DL\plots\confusion_matrix_bilstm_attention.png
📊 Saved: c:\Users\USER\Documents\Tubes NLP\pba2026-kelompok-3\DL\plots\training_curves_bilstm_attention.png


In [16]:
# ======================
# MODEL
# ======================
model = BiLSTMClassifier(
    vocab_size=len(vocab.word2idx),
    embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM
)

# ======================
# TRAINING (INI YANG KAMU TANYA)
# ======================
history = train_model(
    model,
    train_loader,
    val_loader,
    test_loader,
    save_path=BILSTM_MODEL_PATH,
    epochs=LSTM_EPOCHS,
    lr=LSTM_LR,
    patience=LSTM_PATIENCE
)



🚀 Training model...

Epoch 1
Train Loss: 0.6938 | Acc: 0.5158
Val   Loss: 0.6901 | Acc: 0.5500
Epoch 2
Train Loss: 0.6825 | Acc: 0.5667
Val   Loss: 0.6797 | Acc: 0.5630
Epoch 3
Train Loss: 0.6606 | Acc: 0.6075
Val   Loss: 0.6469 | Acc: 0.6360

📊 Evaluasi model...

📄 Classification Report:
              precision    recall  f1-score   support

    negative       0.59      0.78      0.68      1001
    positive       0.68      0.47      0.55      1000

    accuracy                           0.62      2001
   macro avg       0.64      0.62      0.61      2001
weighted avg       0.64      0.62      0.61      2001

Accuracy: 0.6241879060469765
F1 Score: 0.6147921033891701
📊 Saved: c:\Users\USER\Documents\Tubes NLP\pba2026-kelompok-3\DL\plots\confusion_matrix_model.png
📊 Saved: c:\Users\USER\Documents\Tubes NLP\pba2026-kelompok-3\DL\plots\training_curves_model.png


## 📘 9. Kesimpulan
📌 Penjelasan
BiLSTM mampu memahami urutan kata dalam kalimat
BiLSTM + Attention biasanya memberikan performa lebih baik karena fokus pada kata penting
Dataset yang digunakan adalah IMDB Review dengan klasifikasi sentimen (positif/negatif)